# XGBoost on a Colab GPU: last 30 minutes, excess return vs country (QRT × ETH, 24 Nov 2026)

The Colab version of `xgb_last30_kaggle.ipynb`, with the same features and the same CV. It adds:
* **A dataset download** from Kaggle inside the notebook.
* **PyTorch on the GPU** for the heavy parts: price cleaning, all path features (incl.
  beta/idio vs the country path) and the per-round metrics run as CUDA tensors. XGBoost trains on
  the GPU too (`device="cuda"`); the trees use XGBoost's own CUDA kernels, which can't be written
  in PyTorch.
* **Hyperparameter tuning:** a random search scored by 5-fold date-block CV weighted accuracy.
  The best parameters then go through the monitored run (metrics after every round), CV, refit
  and submission.
* **Training matched to the metric** (issue #43): `WEIGHT_POWER` (|y|^p weights),
  `DROP_SMALL_Q` (fit without near-zero moves), `REG_TARGET = "vol"` (y / vol) and `REG_LOSS = "huber"`.
* **Seed bagging** (issue #43): `N_SEEDS` models per fold and in the refit, averaged. The CV
  reports bag vs single seed, paired on the same folds.
* **Outputs saved to Google Drive**, so they survive the Colab session.

**Before you run**
1. *Runtime → Change runtime type → GPU* (T4 is enough).
2. Kaggle credentials, in one of these ways:
   * Colab **Secrets** (key icon on the left): `KAGGLE_API_TOKEN` (new-style token), **or**
     `KAGGLE_USERNAME` + `KAGGLE_KEY`, with notebook access switched on; **or**
   * upload `kaggle.json` when the download cell asks for it (kaggle.com → Settings → API →
     Create New Token).
3. Accept the competition rules once on kaggle.com (otherwise the download fails with 403).
4. *Runtime → Run all.* Allow the Google Drive prompt if `SAVE_TO_DRIVE` is on.

It also runs locally (CPU) from the repo, using `data/raw/**/x_train.parquet`.

In [ ]:
# ---- configuration -------------------------------------------------------------------------
import os

QUICK = os.environ.get("QUICK", "0") == "1"   # smoke test: tiny tuning, few rounds
SEED = 42
SLUG = "qrt-x-eth-hackathon-24-11-2026"
SAVE_TO_DRIVE = True        # Colab: write outputs to MyDrive/qrt_xgb/<timestamp>/
N_FOLDS = 5                 # date-block folds, identical to the team CV (competition.toml)
VAL_FOLD = 4                # the fold monitored round by round
OBJECTIVE = os.environ.get("OBJECTIVE", "binary")   # "binary" (|y|-weighted P(y>0)) | "regression"
CLIP_SIGMA = 5.0            # bar returns beyond median ± CLIP_SIGMA * robust sigma are clipped
WEIGHT_CAP_Q = 0.99         # binary: |y| training weights winsorised at this quantile
USE_STOCK_PRIOR = os.environ.get("USE_STOCK_PRIOR", "0") == "1"   # OOF stock prior (issue #13)
PRIOR_STRENGTH = 50.0

# matching the metric (issue #43)
WEIGHT_POWER = 1.0          # binary: training weight = |y| ** WEIGHT_POWER (try 0.5 / 1 / 1.5)
DROP_SMALL_Q = 0.0          # train without the smallest DROP_SMALL_Q share of |y| (e.g. 0.2); scoring keeps all rows
REG_TARGET = "raw"          # regression target: "raw" = y | "vol" = y / intraday vol (sign unchanged)
REG_LOSS = "squared"        # regression loss: "squared" | "huber" (pseudo-Huber, slope = 1 target std)
# variance reduction (issue #43)
N_SEEDS = 5                 # XGBoost models per fold and in the refit, averaged (1 = off); tuning uses 1

# tuning (random search; trial 0 is PARAMS itself)
RUN_TUNING = os.environ.get("RUN_TUNING", "1") == "1"
N_TRIALS = 30
TUNE_MINUTES = 40           # stop starting new trials after this many minutes
TUNE_ROUNDS = 2000
TUNE_EARLY_STOP = 100

# final model
N_ROUNDS = 3000
EARLY_STOP = 200
ES_METRIC = "wlogloss" if OBJECTIVE == "binary" else "rmse"
PRINT_EVERY = 1             # print the validation metrics every N rounds (1 = every round)
RUN_CV = True
FULL_REFIT = True
REFIT_FACTOR = 1.1

PARAMS = {   # starting point = the Kaggle notebook's parameters (CV 0.54054 ± 0.00399)
    "eta": 0.02, "max_depth": 4, "min_child_weight": 50, "subsample": 0.7,
    "colsample_bytree": 0.6, "reg_lambda": 10.0, "reg_alpha": 0.0, "max_bin": 256,
    "tree_method": "hist", "seed": SEED,
}
if QUICK:
    N_TRIALS, TUNE_ROUNDS, TUNE_EARLY_STOP, SAVE_TO_DRIVE = 2, 30, 10, False
    N_ROUNDS, EARLY_STOP, RUN_CV, PRINT_EVERY, N_SEEDS = 40, 15, False, 5, 2

## 0. Environment, Kaggle download, output folder

In [ ]:
import json
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import xgboost as xgb

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DT = torch.float64                      # features in float64 (same numbers as the Kaggle notebook)
PARAMS["device"] = "cuda" if DEV.type == "cuda" else "cpu"
print(f"colab {IN_COLAB} | torch {torch.__version__} | xgboost {xgb.__version__} | device {DEV}")
if DEV.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
elif IN_COLAB:
    print("WARNING: no GPU. Runtime -> Change runtime type -> GPU, then rerun.")

In [ ]:
def find_local_data() -> Path | None:
    roots = [Path("/content/data"), Path("/kaggle/input")]
    roots += [p / "data" / "raw" for p in [Path.cwd(), *Path.cwd().parents]]
    for root in roots:
        if not root.exists():
            continue
        hits = [h for h in root.rglob("x_train.parquet") if (h.parent / "y_train.csv").exists()]
        if hits:
            return sorted(hits, key=lambda h: (SLUG not in str(h), len(h.parts)))[0].parent
    return None


def kaggle_credentials() -> None:
    # Never hard-code credentials: Colab secrets, an existing ~/.kaggle file, or an upload.
    home = Path.home() / ".kaggle"
    has_env = os.environ.get("KAGGLE_API_TOKEN") or (
        os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))
    if has_env or (home / "kaggle.json").exists() or (home / "access_token").exists():
        return
    if IN_COLAB:
        from google.colab import userdata
        for name in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
            try:
                value = userdata.get(name)
            except Exception:          # secret missing or notebook access not granted
                value = None
            if value:
                os.environ[name] = value
        if os.environ.get("KAGGLE_API_TOKEN") or (
                os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
            return
        from google.colab import files
        print("No Kaggle secret found. Upload kaggle.json (or a file holding your API token):")
        uploaded = files.upload()
        name, content = next(iter(uploaded.items()))
        home.mkdir(exist_ok=True)
        target = home / ("kaggle.json" if name.endswith(".json") else "access_token")
        target.write_bytes(content)
        target.chmod(0o600)
        return
    raise RuntimeError("Kaggle credentials missing: set KAGGLE_API_TOKEN or ~/.kaggle/kaggle.json")


def download_competition(dest: Path) -> Path:
    if IN_COLAB:   # Colab's preinstalled kaggle CLI can be too old for new-style tokens
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kaggle"], check=True)
    kaggle_credentials()
    dest.mkdir(parents=True, exist_ok=True)
    cli = shutil.which("kaggle") or "kaggle"
    subprocess.run([cli, "competitions", "download", "-c", SLUG, "-p", str(dest)], check=True)
    for archive in dest.glob("*.zip"):
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(dest)
        archive.unlink()
    return dest


DATA_DIR = find_local_data() or download_competition(Path("/content/data") if IN_COLAB else Path("data_kaggle"))
print("data:", DATA_DIR, sorted(p.name for p in DATA_DIR.iterdir()))

In [ ]:
def make_out_dir() -> Path:
    if os.environ.get("OUT_DIR"):
        return Path(os.environ["OUT_DIR"])
    stamp = time.strftime("%m%d-%H%M%S")
    if IN_COLAB and SAVE_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        return Path("/content/drive/MyDrive/qrt_xgb") / stamp
    if IN_COLAB:
        return Path("/content/outputs") / stamp
    repo = next((p for p in DATA_DIR.parents if (p / "pyproject.toml").exists()), Path.cwd())
    return repo / "runs" / f"efe-claude-1_xgb_colab_{stamp}"     # runs/ is gitignored


OUT_DIR = make_out_dir()
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("out:", OUT_DIR)

In [ ]:
# ---- load ----------------------------------------------------------------------------------
TARGET = "Close return"
KEYS = ["ID", "DATE_ID", "STOCK_ID", "COUNTRY"]
x_train = pd.read_parquet(DATA_DIR / "x_train.parquet")
x_test = pd.read_parquet(DATA_DIR / "x_test.parquet")
y_train = pd.read_csv(DATA_DIR / "y_train.csv")
raw = pd.concat([x_train.assign(is_train=True), x_test.assign(is_train=False)], ignore_index=True)
raw = raw.merge(y_train, on="ID", how="left")
print(f"train {len(x_train):,} rows / {x_train.DATE_ID.nunique()} days | "
      f"test {len(x_test):,} rows / {x_test.DATE_ID.nunique()} days | stocks {raw.STOCK_ID.nunique()}")
del x_train, x_test

## 1. Denoise the price paths (PyTorch, GPU)

Paths are right-aligned so the last column is the cutoff. Gaps are forward- then back-filled; the
padding of short (half) days stays empty. Bar log returns are clipped at median ± `CLIP_SIGMA`
robust (MAD) sigmas of their own row, which removes bad ticks. The raw last bar is kept for the
benchmark signal.

In [ ]:
WIDTH = 96
EPS = 1e-12


def tensor(a) -> torch.Tensor:
    return torch.as_tensor(np.asarray(a), dtype=DT, device=DEV)


def nanstd(x: torch.Tensor, keepdim: bool = False) -> torch.Tensor:
    mean = torch.nanmean(x, dim=1, keepdim=True)
    return torch.nanmean((x - mean) ** 2, dim=1, keepdim=keepdim).sqrt()


def fill_forward(x: torch.Tensor) -> torch.Tensor:
    cols = torch.arange(x.shape[1], device=x.device).expand_as(x)
    idx = torch.where(~torch.isnan(x), cols, torch.zeros_like(cols)).cummax(dim=1).values
    return x.gather(1, idx)


def price_matrix(series: pd.Series) -> tuple[torch.Tensor, torch.Tensor]:
    out = np.full((len(series), WIDTH), np.nan)
    n_bars = np.zeros(len(series), dtype=np.int64)
    for i, arr in enumerate(series.to_numpy()):
        arr = np.asarray(arr, dtype=float)[-WIDTH:]
        out[i, WIDTH - len(arr):] = arr
        n_bars[i] = len(arr)
    p = tensor(out)
    p = torch.where(p > 0, p, torch.full_like(p, float("nan")))
    n = torch.as_tensor(n_bars, device=DEV)
    pad = torch.arange(WIDTH, device=DEV)[None, :] < (WIDTH - n)[:, None]
    p = fill_forward(p).flip(1)
    p = fill_forward(p).flip(1)                     # back-fill a missing first price
    p[pad] = float("nan")
    return p, n


def clean_bars(prices: torch.Tensor, clip_sigma: float) -> tuple[torch.Tensor, torch.Tensor]:
    raw_r = torch.diff(torch.log(prices), dim=1)
    med = torch.nanmedian(raw_r, dim=1, keepdim=True).values
    mad = 1.4826 * torch.nanmedian((raw_r - med).abs(), dim=1, keepdim=True).values
    sigma = torch.where(mad > 0, mad, nanstd(raw_r, keepdim=True))
    sigma = torch.where(sigma > 0, sigma, torch.full_like(sigma, 1e-6))
    return raw_r, torch.clamp(raw_r, min=med - clip_sigma * sigma, max=med + clip_sigma * sigma)


t0 = time.time()
PRICES, N_BARS = price_matrix(raw["Price TS"])
RAW_R, R = clean_bars(PRICES, CLIP_SIGMA)
valid = ~torch.isnan(RAW_R)
print(f"bars clipped: {float((RAW_R != R)[valid].double().sum() / RAW_R.numel()):.3%} | "
      f"NaN bars: {float(torch.isnan(R).double().mean()):.3%} | {time.time() - t0:.1f}s on {DEV}")

## 2. Features (PyTorch, GPU), then cross-sections (pandas)

These are the same features as the Kaggle notebook: vol-scaled returns `z_k`, trend slopes, EMA
deviation, autocorrelation, intraday shape, beta/idiosyncratic moves against the equal-weight
country path, then demeaned values and ranks within `DATE_ID × COUNTRY`, day context, and the
stock's own typical values (no target).

In [ ]:
LAGS = (1, 2, 3, 6, 12, 24, 48)


def slope(path: torch.Tensor, k: int) -> torch.Tensor:
    y = path[:, -k:]
    x = torch.arange(k, device=DEV, dtype=DT) - (k - 1) / 2
    return ((y - y.mean(dim=1, keepdim=True)) * x).sum(dim=1) / (x ** 2).sum()


def ema_last(path: torch.Tensor, span: int) -> torch.Tensor:
    alpha, ema = 2 / (span + 1), path[:, 0].clone()
    for j in range(1, path.shape[1]):
        ema = alpha * path[:, j] + (1 - alpha) * ema
    return ema


def row_features(r: torch.Tensor, raw_r: torch.Tensor, n_bars: torch.Tensor) -> dict:
    n = n_bars.to(DT)
    vol = nanstd(r) + EPS
    path = torch.cumsum(torch.nan_to_num(r, nan=0.0), dim=1)    # cleaned log path, starts at 0
    last = path[:, -1]
    f = {"n_bars": n, "vol": vol, "zero_frac": (raw_r == 0).to(DT).mean(dim=1),
         "vol_last12": nanstd(r[:, -12:])}
    f["vol_ratio"] = f["vol_last12"] / vol
    for k in LAGS:
        f[f"ret_{k}"] = torch.nansum(r[:, -k:], dim=1)
        f[f"z_{k}"] = f[f"ret_{k}"] / (vol * k ** 0.5)
    f["ret_day"] = last
    f["z_day"] = last / (vol * torch.clamp(n - 1, min=1).sqrt())
    f["last_bar_z"] = torch.nan_to_num(raw_r[:, -1], nan=0.0) / vol
    for k in (12, 24, 48):
        f[f"slope_{k}"] = slope(path, k) / vol
    for span in (6, 24):
        f[f"ema_dev_{span}"] = (last - ema_last(path, span)) / vol
    tail = r[:, -48:] - torch.nanmean(r[:, -48:], dim=1, keepdim=True)
    f["ac1"] = torch.nansum(tail[:, 1:] * tail[:, :-1], dim=1) / (torch.nansum(tail ** 2, dim=1) + EPS)
    hi, lo = path.max(dim=1).values, path.min(dim=1).values
    f["range_pos"] = (last - lo) / (hi - lo + EPS)
    f["drawdown"] = (last - hi) / vol
    f["runup"] = (last - lo) / vol
    return f


def group_mean(x: torch.Tensor, gid: torch.Tensor, n_groups: int) -> torch.Tensor:
    ok = ~torch.isnan(x)
    sums = torch.zeros(n_groups, x.shape[1], dtype=DT, device=DEV).index_add_(0, gid, torch.where(ok, x, 0.0))
    cnts = torch.zeros(n_groups, x.shape[1], dtype=DT, device=DEV).index_add_(0, gid, ok.to(DT))
    return (sums / cnts)[gid]                          # NaN where a group has no data in a bar


def country_features(r: torch.Tensor, keys: pd.DataFrame, n_bars: torch.Tensor) -> dict:
    codes, uniq = pd.factorize(keys["DATE_ID"].astype(str) + "|" + keys["COUNTRY"].astype(str))
    m = group_mean(r, torch.as_tensor(codes, device=DEV), len(uniq))   # equal-weight country bars
    rc = r - torch.nanmean(r, dim=1, keepdim=True)
    mc = m - torch.nanmean(m, dim=1, keepdim=True)
    cov, var_m = torch.nanmean(rc * mc, dim=1), torch.nanmean(mc ** 2, dim=1) + EPS
    beta = cov / var_m
    idio = r - beta[:, None] * m
    idio_vol = nanstd(idio) + EPS
    n = n_bars.to(DT)
    f = {"beta": beta, "corr_cty": cov / (var_m * (torch.nanmean(rc ** 2, dim=1) + EPS)).sqrt(),
         "idio_vol": idio_vol, "idio_day": torch.nansum(idio, dim=1)}
    f["idio_z_day"] = f["idio_day"] / (idio_vol * torch.clamp(n - 1, min=1).sqrt())
    for k in (1, 3, 6, 12):
        f[f"idio_z_{k}"] = torch.nansum(idio[:, -k:], dim=1) / (idio_vol * k ** 0.5)
    return f


CS_COLS = ["z_1", "z_3", "z_6", "z_12", "z_48", "z_day", "last_bar_z", "slope_12", "slope_48",
           "ema_dev_6", "ema_dev_24", "vol", "vol_ratio", "range_pos", "idio_z_day", "idio_z_6"]


def cross_section(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby(["DATE_ID", "COUNTRY"])
    rel = (df[CS_COLS] - g[CS_COLS].transform("mean")).add_suffix("_rel")
    rank = g[CS_COLS].rank(pct=True).add_suffix("_rank")
    ctx = pd.DataFrame({
        "cty_ret_day": g["ret_day"].transform("mean"), "cty_ret_6": g["ret_6"].transform("mean"),
        "cty_disp_day": g["ret_day"].transform("std"), "cty_vol": g["vol"].transform("mean"),
        "cty_n": g["ret_day"].transform("size")}, index=df.index)
    s = df.groupby("STOCK_ID")
    stock = pd.DataFrame({
        "vol_vs_stock": df["vol"] / s["vol"].transform("median"),
        "z6_vs_stock": df["z_6"] - s["z_6"].transform("mean"),
        "last_bar_vs_stock": df["last_bar_z"] - s["last_bar_z"].transform("mean")}, index=df.index)
    return pd.concat([rel, rank, ctx, stock], axis=1)


t0 = time.time()
gpu_feats = {**row_features(R, RAW_R, N_BARS), **country_features(R, raw[KEYS], N_BARS)}
feats = pd.DataFrame({k: v.cpu().numpy() for k, v in gpu_feats.items()})
df = pd.concat([raw[KEYS + ["is_train", TARGET]], feats], axis=1)
df["country_code"] = df["COUNTRY"].astype("category").cat.codes
df["open_min"] = raw["OPEN_TIME"].str.slice(0, 2).astype(int) * 60 + raw["OPEN_TIME"].str.slice(3, 5).astype(int)
df = pd.concat([df, cross_section(df)], axis=1)
df = df.replace([np.inf, -np.inf], np.nan)
FEATURES = [c for c in df.columns if c not in KEYS + ["is_train", TARGET]]
print(f"{len(FEATURES)} features in {time.time() - t0:.1f}s")
del PRICES, RAW_R, R, gpu_feats, feats
if DEV.type == "cuda":
    torch.cuda.empty_cache()

## 3. Target, weights, folds, optional stock prior

In [ ]:
train_df = df[df["is_train"]].reset_index(drop=True)
Y = train_df[TARGET].to_numpy()
DATES = train_df["DATE_ID"].to_numpy()


def date_block_folds(dates: np.ndarray, n_splits: int) -> list[tuple[np.ndarray, np.ndarray]]:
    uniq = np.sort(pd.unique(dates))
    pos = np.searchsorted(uniq, dates)
    folds = []
    for block in np.array_split(np.arange(len(uniq)), n_splits):
        val = (pos >= block[0]) & (pos <= block[-1])
        folds.append((np.flatnonzero(~val), np.flatnonzero(val)))
    return folds


VOL = train_df["vol"].to_numpy()
TARGET_SCALE = {"raw": float(Y.std()), "vol": float((Y / VOL).std())}   # fixed for every fold


def fit_target(y: np.ndarray, vol: np.ndarray) -> tuple[np.ndarray, np.ndarray | None]:
    # Binary: label sign(y), weight |y|^p winsorised. Regression: y or y/vol, scaled to unit std.
    if OBJECTIVE == "binary":
        w = np.abs(y) ** WEIGHT_POWER
        w = np.minimum(w, np.quantile(w, WEIGHT_CAP_Q))
        return (y > 0).astype(np.float32), (w / w.mean()).astype(np.float32)
    t = y / vol if REG_TARGET == "vol" else y
    return (t / TARGET_SCALE[REG_TARGET]).astype(np.float32), None


def fit_rows(idx: np.ndarray) -> np.ndarray:
    # Rows used to fit: drop the smallest |y| (near-zero moves are noise); validation keeps all.
    if DROP_SMALL_Q <= 0:
        return idx
    a = np.abs(Y[idx])
    return idx[a >= np.quantile(a, DROP_SMALL_Q)]


def objective() -> dict:
    if OBJECTIVE == "binary":
        return {"objective": "binary:logistic"}
    if REG_LOSS == "huber":
        return {"objective": "reg:pseudohubererror", "huber_slope": 1.0}
    return {"objective": "reg:squarederror"}


def stock_prior(frame: pd.DataFrame, fit_idx: np.ndarray) -> np.ndarray:
    stock, y = frame["STOCK_ID"].to_numpy(), frame[TARGET].to_numpy()

    def prior_map(idx):
        d = pd.DataFrame({"s": stock[idx], "y": y[idx], "a": np.abs(y[idx])}).groupby("s").sum()
        return d["y"] / (d["a"] + PRIOR_STRENGTH * np.abs(y[idx]).mean())

    out = pd.Series(stock).map(prior_map(fit_idx)).to_numpy(dtype=float)
    for tr, va in date_block_folds(frame["DATE_ID"].to_numpy()[fit_idx], N_FOLDS):
        out[fit_idx[va]] = pd.Series(stock[fit_idx[va]]).map(prior_map(fit_idx[tr])).to_numpy()
    return np.nan_to_num(out)


def design(frame: pd.DataFrame, fit_idx: np.ndarray) -> pd.DataFrame:
    X = frame[FEATURES].astype(np.float32)
    if USE_STOCK_PRIOR:
        X = X.assign(stock_prior=stock_prior(frame, fit_idx).astype(np.float32))
    return X


FOLDS = date_block_folds(DATES, N_FOLDS)
print("fold sizes (train / val rows):", [(len(a), len(b)) for a, b in FOLDS])

## 4. GPU metrics after every boosting round

`weighted_accuracy` = Σ |y| · [sign(pred) = sign(y)] / Σ |y|, which is the competition metric.
The metrics are computed in PyTorch on the GPU. The weighted AUC is exact, including ties.

In [ ]:
THRESHOLD = 0.5 if OBJECTIVE == "binary" else 0.0


def weighted_auc(score: torch.Tensor, pos: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
    order = torch.argsort(score)
    score, pos, w = score[order], pos[order].to(DT), w[order]
    _, inv = torch.unique_consecutive(score, return_inverse=True)
    n = int(inv[-1]) + 1
    w_pos = torch.zeros(n, dtype=DT, device=DEV).index_add_(0, inv, w * pos)
    w_neg = torch.zeros(n, dtype=DT, device=DEV).index_add_(0, inv, w * (1 - pos))
    below = torch.cumsum(w_neg, 0) - w_neg
    return (w_pos * (below + 0.5 * w_neg)).sum() / (w_pos.sum() * w_neg.sum())


def metrics(score, y, label=None) -> dict:
    s, y = tensor(score), tensor(y)
    w, pos = y.abs(), y > 0
    correct = (s > THRESHOLD) == pos
    out = {"wacc": (w * correct).sum() / w.sum(), "acc": correct.to(DT).mean(),
           "wauc": weighted_auc(s, pos, w)}
    if OBJECTIVE == "binary":
        p = s.clamp(1e-7, 1 - 1e-7)
        out["wlogloss"] = -(w * torch.where(pos, p.log(), (1 - p).log())).sum() / w.sum()
    elif label is not None:                 # RMSE on the fitted (scaled) regression target
        out["rmse"] = ((s - tensor(label)) ** 2).mean().sqrt()
    return {k: float(v) for k, v in out.items()}


SIGNED_Y: dict[int, np.ndarray] = {}         # id(DMatrix) -> true signed target


def feval(predt: np.ndarray, dmat: xgb.DMatrix) -> list[tuple[str, float]]:
    # With a built-in objective XGBoost passes transformed predictions (probabilities).
    m = metrics(predt, SIGNED_Y[id(dmat)], dmat.get_label())
    order = [k for k in m if k != ES_METRIC] + [ES_METRIC]    # early stopping uses ES_METRIC
    return [(k, m[k]) for k in order]


class RoundLogger(xgb.callback.TrainingCallback):
    def __init__(self, every: int, verbose: bool = True):
        self.every, self.verbose, self.rows, self.t0 = every, verbose, [], time.time()

    def after_iteration(self, model, epoch, evals_log) -> bool:
        row = {"round": epoch + 1}
        for data, ms in evals_log.items():
            row.update({f"{data}_{k}": v[-1] for k, v in ms.items()})
        self.rows.append(row)
        if self.verbose and (epoch == 0 or (epoch + 1) % self.every == 0):
            val = "  ".join(f"{k.removeprefix('valid_')} {v:.5f}" for k, v in row.items()
                            if k.startswith("valid_"))
            print(f"[{epoch + 1:5d}] {val}  | train wacc {row.get('train_wacc', np.nan):.5f}"
                  f"  ({time.time() - self.t0:.0f}s)")
        return False


def make_dmatrix(X: pd.DataFrame, y: np.ndarray, fit_weights: bool, params: dict, ref=None):
    label, w = fit_target(y, X["vol"].to_numpy())
    kwargs = {"label": label, "weight": w if fit_weights else None, "ref": ref,
              "max_bin": params["max_bin"], "feature_names": list(X.columns)}
    data = torch.as_tensor(X.to_numpy(np.float32), device=DEV)   # GPU-resident input
    try:
        d = xgb.QuantileDMatrix(data, **kwargs)
    except (TypeError, ValueError):     # XGBoost build without torch/CUDA-array input support
        d = xgb.QuantileDMatrix(X.to_numpy(np.float32), **kwargs)
    SIGNED_Y[id(d)] = y
    return d


def train_fold(tr, va, params, n_rounds, early_stop, verbose=False, track_train=False, n_seeds=1):
    # Trains n_seeds boosters (seed, seed+1, ...) and averages them. Seed 0 is the one logged
    # round by round. Returns (boosters, seed-0 history, bagged prediction, seed-0 prediction).
    X = design(train_df, tr)
    fit = fit_rows(tr)
    dtr = make_dmatrix(X.iloc[fit], Y[fit], True, params)
    dva = make_dmatrix(X.iloc[va], Y[va], False, params, ref=dtr)
    boosters, preds, history = [], [], None
    for s in range(n_seeds):
        evals = [(dtr, "train"), (dva, "valid")] if track_train and s == 0 else [(dva, "valid")]
        full = {**params, **objective(), "seed": params["seed"] + s, "disable_default_eval_metric": 1}
        logger = RoundLogger(PRINT_EVERY, verbose and s == 0)
        stop = xgb.callback.EarlyStopping(rounds=early_stop, metric_name=ES_METRIC, data_name="valid",
                                          maximize=ES_METRIC in ("wacc", "acc", "wauc"), save_best=True)
        booster = xgb.train(full, dtr, num_boost_round=n_rounds, evals=evals, custom_metric=feval,
                            callbacks=[logger, stop], verbose_eval=False)
        boosters.append(booster)
        preds.append(booster.predict(dva))
        if s == 0:
            history = pd.DataFrame(logger.rows)
        if verbose and n_seeds > 1:
            print(f"seed {s}: best round {booster.best_iteration + 1}, "
                  f"wacc {metrics(preds[-1], Y[va])['wacc']:.5f}, bag so far "
                  f"{metrics(np.mean(preds, axis=0), Y[va])['wacc']:.5f}")
    return boosters, history, np.mean(preds, axis=0), preds[0]


def mean_best_round(boosters) -> int:
    return int(np.mean([b.best_iteration + 1 for b in boosters]))

## 5. Hyperparameter tuning on the GPU (random search, 5-fold CV)

Trial 0 is the starting `PARAMS`. Every trial trains all 5 folds with early stopping on
`ES_METRIC` and is scored by the mean CV weighted accuracy. Progress is saved after each trial
(`tuning_results.csv`). Picking the best of many noisy trials flatters its score a little, so
trust the gain only if it is clearly larger than the fold std.

In [ ]:
def sample_params(rng: np.random.Generator) -> dict:
    def log_uniform(lo, hi):
        return float(np.exp(rng.uniform(np.log(lo), np.log(hi))))

    return {**PARAMS, "eta": log_uniform(0.01, 0.1), "max_depth": int(rng.choice([3, 4, 5, 6, 7])),
            "min_child_weight": log_uniform(5, 500), "subsample": float(rng.uniform(0.5, 0.95)),
            "colsample_bytree": float(rng.uniform(0.3, 0.9)), "reg_lambda": log_uniform(0.5, 100),
            "reg_alpha": log_uniform(1e-3, 10), "gamma": log_uniform(1e-4, 1.0)}


def cv_score(params: dict, folds=FOLDS) -> dict:
    scores, rounds = [], []
    for tr, va in folds:
        boosters, _, pred, _ = train_fold(tr, va, params, TUNE_ROUNDS, TUNE_EARLY_STOP)
        scores.append(metrics(pred, Y[va])["wacc"])
        rounds.append(mean_best_round(boosters))
    return {"cv_wacc": float(np.mean(scores)), "cv_std": float(np.std(scores)),
            "rounds": int(np.mean(rounds))}


TUNED = dict(PARAMS)
if RUN_TUNING:
    rng = np.random.default_rng(SEED)
    tune_folds = FOLDS if not QUICK else FOLDS[:2]
    trials, t_start = [], time.time()
    for t in range(N_TRIALS):
        if (time.time() - t_start) / 60 > TUNE_MINUTES:
            print(f"time budget reached after {t} trials")
            break
        params = dict(PARAMS) if t == 0 else sample_params(rng)
        t0 = time.time()
        res = cv_score(params, tune_folds)
        trials.append({"trial": t, **res, **{k: params[k] for k in params if k not in ("device", "tree_method", "seed")},
                       "seconds": round(time.time() - t0, 1)})
        best = max(trials, key=lambda r: r["cv_wacc"])
        print(f"trial {t:2d}: cv {res['cv_wacc']:.5f} ± {res['cv_std']:.5f}  rounds {res['rounds']:4d}  "
              f"eta {params['eta']:.3f} depth {params['max_depth']} mcw {params['min_child_weight']:.0f}  "
              f"({time.time() - t0:.0f}s) | best so far {best['cv_wacc']:.5f} (trial {best['trial']})")
        pd.DataFrame(trials).to_csv(OUT_DIR / "tuning_results.csv", index=False)
    best = max(trials, key=lambda r: r["cv_wacc"])
    TUNED = {**PARAMS, **{k: best[k] for k in PARAMS if k in best}}
    for k in ("gamma",):
        if k in best:
            TUNED[k] = best[k]
    TUNED["max_depth"] = int(TUNED["max_depth"])
    print(f"\nbest trial {best['trial']}: cv {best['cv_wacc']:.5f} ± {best['cv_std']:.5f}"
          f" (trial 0 = starting PARAMS: {trials[0]['cv_wacc']:.5f})")
print("parameters used below:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in TUNED.items()})

## 6. Monitored training with the tuned parameters (metrics every round)

In [ ]:
tr, va = FOLDS[VAL_FOLD]
holdout_models, history, holdout_pred, holdout_single = train_fold(
    tr, va, TUNED, N_ROUNDS, EARLY_STOP, verbose=True, track_train=True, n_seeds=N_SEEDS)
best_round = mean_best_round(holdout_models)          # used for the refit
plot_round = holdout_models[0].best_iteration + 1     # seed 0, the logged curve
holdout = metrics(holdout_pred, Y[va])
history.to_csv(OUT_DIR / "history_holdout.csv", index=False)
for s, m in enumerate(holdout_models):
    m.save_model(OUT_DIR / f"model_holdout_seed{s}.json")
print(f"\nsingle seed wacc {metrics(holdout_single, Y[va])['wacc']:.5f} -> "
      f"bag of {N_SEEDS} wacc {holdout['wacc']:.5f}")

bench = -train_df["last_bar_z"].to_numpy()[va]        # host benchmark: -(last 5-minute return)
w_va = np.abs(Y[va])
bench_wacc = (w_va * ((bench > 0) == (Y[va] > 0))).sum() / w_va.sum()
print(f"\nbest round {best_round} | holdout {json.dumps({k: round(v, 5) for k, v in holdout.items()})}")
print(f"benchmark -(last bar) on the same days: wacc {bench_wacc:.5f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ["wacc", ES_METRIC], strict=True):
    for split in ("train", "valid"):
        ax.plot(history["round"], history[f"{split}_{metric}"], label=split)
    ax.axvline(plot_round, color="grey", ls="--", lw=1, label="best round (seed 0)")
    ax.set_title(metric)
    ax.set_xlabel("boosting round")
    ax.legend()
axes[0].axhline(bench_wacc, color="red", ls=":", lw=1, label="benchmark")
plt.tight_layout()
plt.show()

day_vol = train_df.iloc[va].groupby("DATE_ID")["vol"].transform("mean").to_numpy()
calm = day_vol <= np.median(day_vol)      # test days are calmer (data audit, PR #15)
for name, mask in (("calm days", calm), ("volatile days", ~calm)):
    print(f"{name:14s} wacc {metrics(holdout_pred[mask], Y[va][mask])['wacc']:.5f}")

## 7. CV over all 5 folds with the tuned parameters

In [ ]:
oof, oof_single = np.full(len(train_df), np.nan), np.full(len(train_df), np.nan)
oof[va], oof_single[va] = holdout_pred, holdout_single
fold_scores = {VAL_FOLD: holdout["wacc"]}
single_scores = {VAL_FOLD: metrics(holdout_single, Y[va])["wacc"]}
best_rounds = {VAL_FOLD: best_round}
if RUN_CV:
    for k, (tr_k, va_k) in enumerate(FOLDS):
        if k == VAL_FOLD:
            continue
        t0 = time.time()
        models_k, _, pred_k, single_k = train_fold(tr_k, va_k, TUNED, N_ROUNDS, EARLY_STOP, n_seeds=N_SEEDS)
        oof[va_k], oof_single[va_k] = pred_k, single_k
        fold_scores[k] = metrics(pred_k, Y[va_k])["wacc"]
        single_scores[k] = metrics(single_k, Y[va_k])["wacc"]
        best_rounds[k] = mean_best_round(models_k)
        print(f"fold {k}: bag wacc {fold_scores[k]:.5f} | single seed {single_scores[k]:.5f} | "
              f"best round {best_rounds[k]} | {time.time() - t0:.0f}s")

folds_done = sorted(fold_scores)
scores = np.array([fold_scores[k] for k in folds_done])
singles = np.array([single_scores[k] for k in folds_done])
gain = scores - singles
done = ~np.isnan(oof)
settings = {k: globals()[k] for k in ("OBJECTIVE", "WEIGHT_POWER", "DROP_SMALL_Q", "REG_TARGET",
                                      "REG_LOSS", "N_SEEDS", "USE_STOCK_PRIOR")}
cv = {"settings": settings, "params": TUNED, "fold_wacc": fold_scores, "fold_wacc_single_seed": single_scores,
      "best_rounds": best_rounds, "cv_wacc_mean": float(scores.mean()), "cv_wacc_std": float(scores.std()),
      "cv_wacc_single_seed_mean": float(singles.mean()),
      "bagging_gain_mean": float(gain.mean()), "bagging_gain_std": float(gain.std()),
      "pooled_oof": metrics(oof[done], Y[done])}
print(f"\nweighted accuracy (bag of {N_SEEDS}): {cv['cv_wacc_mean']:.5f} ± {cv['cv_wacc_std']:.5f} "
      f"over {len(scores)} fold(s)   (Kaggle notebook default: 0.54054 ± 0.00399; U003: 0.5440)")
print(f"single seed: {singles.mean():.5f} | bagging gain, paired by fold: "
      f"{gain.mean():+.5f} ± {gain.std():.5f} ({int((gain > 0).sum())}/{len(gain)} folds up)")
print("settings:", settings)
(OUT_DIR / "cv_results.json").write_text(json.dumps(cv, indent=2, default=str))
pd.DataFrame({"ID": train_df["ID"], "DATE_ID": DATES, "y": Y, "oof": oof, "oof_single_seed": oof_single}
             ).to_parquet(OUT_DIR / "oof_predictions.parquet", index=False)

## 8. Refit on all training days, save the model, write the submission

Kaggle expects **every** id (219,157 rows) with `Close return` = ±1. Hand the file to the team's
Submitter instead of submitting it yourself (AGENTS.md §9.1).

In [ ]:
all_idx = np.arange(len(train_df))
if FULL_REFIT:
    n_final = max(1, int(np.mean(list(best_rounds.values())) * REFIT_FACTOR))
    fit = fit_rows(all_idx)
    d_all = make_dmatrix(design(train_df, all_idx).iloc[fit], Y[fit], True, TUNED)
    final_models = []
    for s in range(N_SEEDS):
        full = {**TUNED, **objective(), "seed": TUNED["seed"] + s}
        final_models.append(xgb.train(full, d_all, num_boost_round=n_final))
    print(f"refit {N_SEEDS} seed(s) on {len(fit):,} rows for {n_final} rounds each")
else:
    final_models = holdout_models
for s, m in enumerate(final_models):
    m.save_model(OUT_DIR / f"model_final_seed{s}.json")

X_sub = design(df, np.flatnonzero(df["is_train"].to_numpy()))
d_sub = xgb.DMatrix(X_sub)
score = np.mean([m.predict(d_sub) for m in final_models], axis=0)
fallback = np.where(-df["last_bar_z"].to_numpy() > 0, 1, -1)
signs = np.where(score > THRESHOLD, 1, np.where(score < THRESHOLD, -1, fallback))
sub = pd.DataFrame({"ID": df["ID"].to_numpy(), TARGET: signs}).sort_values("ID")
assert len(sub) == len(df) and sub["ID"].is_unique and set(sub[TARGET]) <= {-1, 1}
sub.to_csv(OUT_DIR / "submission_xgb.csv", index=False)
print(f"submission: {len(sub):,} rows, +1 share on test rows "
      f"{(signs[~df['is_train'].to_numpy()] > 0).mean():.3f} -> {OUT_DIR / 'submission_xgb.csv'}")

importance = pd.concat([pd.Series(m.get_score(importance_type="total_gain")) for m in final_models],
                       axis=1).fillna(0).sum(axis=1).sort_values(ascending=False)
importance.rename("total_gain").to_csv(OUT_DIR / "feature_importance.csv")
print(importance.head(20).round(2).to_string())
config = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, bool, dict))
          and k not in ("SIGNED_Y",)}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=2, default=str))
print("\nsaved:", sorted(p.name for p in OUT_DIR.iterdir()))